In [ ]:

from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

# --- find the data_ingestion dir from wherever we are ---
def find_data_dir():
    for root in (Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]):
        cand = root / "data_ingestion"
        if cand.is_dir():
            return cand
    raise FileNotFoundError("Could not find data_ingestion folder")

DATA = find_data_dir()
print("Using data dir:", DATA)

# --- load both files ---
lyrics = TextLoader(str(DATA / "heylog_12_gauge_lyrics.txt")).load()
para   = TextLoader(str(DATA / "random_paragraph.txt")).load()

# add a "category" field so we can filter by source type later
for d in lyrics:
    d.metadata["category"] = "lyrics"
for d in para:
    d.metadata["category"] = "paragraph"

# --- split into chunks ---
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks   = splitter.split_documents(lyrics + para)

# --- embeddings (local Ollama) ---
emb = OllamaEmbeddings(model="nomic-embed-text")  # 768 dims, local

print(f"{len(chunks)} chunks ready")
print("sample metadata:", chunks[0].metadata)


In [ ]:

from langchain_community.vectorstores import FAISS

# from_documents: bulk ingest chunks + embeddings -> returns a VectorStore
faiss_store = FAISS.from_documents(chunks, emb)
print("FAISS index size:", faiss_store.index.ntotal)


In [ ]:

# similarity_search: returns top-k Document objects
results = faiss_store.similarity_search("dreaming of someone", k=3)

for i, doc in enumerate(results):
    print(f"\n[{i}]")
    print(f"    source: {doc.metadata['source']}")
    print(f"    text:   {doc.page_content[:80]}")


In [ ]:

# similarity_search_with_score: (Document, distance) tuples
# distance = L2 norm -- lower = more similar
scored = faiss_store.similarity_search_with_score("library and silence", k=3)

for doc, score in scored:
    print(f"distance {score:.4f} | {doc.page_content[:60]}")


In [ ]:

# save/load: persist index to disk so you don't re-embed next time
faiss_store.save_local("faiss_index")
print("saved faiss_index/")

reloaded = FAISS.load_local(
    "faiss_index",
    emb,
    allow_dangerous_deserialization=True,  # needed for pickle format
)
print("reloaded, index size:", reloaded.index.ntotal)


In [ ]:

from langchain_community.vectorstores import Chroma

# persist_directory is created automatically
chroma_store = Chroma.from_documents(
    chunks,
    emb,
    persist_directory="chroma_db",
    collection_name="lyrics",
)
print("Chroma collection size:", chroma_store._collection.count())


In [ ]:

# similarity search -- same API as FAISS
results = chroma_store.similarity_search("vivid thoughts and silence", k=2)

for doc in results:
    print(f"source: {doc.metadata['source']}")
    print(f"  {doc.page_content[:70]}\n")


In [ ]:

# similarity search with score
scored = chroma_store.similarity_search_with_score("clothes and wardrobe", k=3)
for doc, score in scored:
    print(f"distance {score:.4f} | {doc.page_content[:60]}")


In [ ]:

# metadata filtering: only search chunks where category == "lyrics"
filtered_store = Chroma.from_documents(
    chunks,
    emb,
    persist_directory="chroma_filtered",
    collection_name="lyrics_filtered",
)

# filter on the "category" metadata field we added during setup
results = filtered_store.similarity_search(
    "dreams and sleeping",
    k=2,
    filter={"category": "lyrics"},  # skip paragraph chunks
)

print("filtered results:")
for doc in results:
    print(f"  [{doc.metadata['category']}] {doc.page_content[:70]}")
